In [1]:
import itertools
import networkx as nx

def is_dominating_broadcast(G, f):
    """ตรวจว่า f เป็น dominating broadcast ตามนิยามใน paper หรือไม่"""
    ecc = nx.eccentricity(G)
    if any(f[v] > ecc[v] for v in G.nodes()):      # เงื่อนไข 1: f(v) <= e(v)
        return False
    dist = dict(nx.all_pairs_shortest_path_length(G))
    for u in G.nodes():                            # เงื่อนไข 2: ทุก u ต้องได้ยิน
        if not any(f[v] >= 1 and dist[v][u] <= f[v] for v in G.nodes()):
            return False
    return True

def gamma_b(G):
    """ brute force หาค่า cost ต่ำสุด = γ_b(G) สำหรับกราฟเล็ก """
    ecc = nx.eccentricity(G)
    nodes = list(G.nodes())
    best = len(nodes)   # ขอบบนตั้งต้น: ใส่ 1 ทุก vertex (cost = n) ใช้ได้เสมอ
    for combo in itertools.product(*[range(ecc[v]+1) for v in nodes]):
        cost = sum(combo)
        if cost < best:                            # ตัดกิ่งที่ถูกกว่าคำตอบปัจจุบัน
            if is_dominating_broadcast(G, dict(zip(nodes, combo))):
                best = cost
    return best

def packing_number(G):
    """ packing ใหญ่สุดที่ทุกคู่ห่างกัน >= 3 ตามนิยาม paper """
    dist = dict(nx.all_pairs_shortest_path_length(G))
    nodes = list(G.nodes())
    for r in range(len(nodes), 0, -1):
        for combo in itertools.combinations(nodes, r):
            if all(dist[u][v] >= 3 for u, v in itertools.combinations(combo, 2)):
                return r
    return 1

In [2]:
tests = [
    ("P_4", nx.path_graph(4),     2, 2),  # ยิงแรง 2 จาก vertex ตัวที่สองคลุมหมด
    ("C_6", nx.cycle_graph(6),    2, 2),  # ยิงแรง 1 จากคู่ตรงข้าม 2 ตัวคลุมหมด
    ("K_4", nx.complete_graph(4), 1, 1),  # ยิงแรง 1 จากตัวเดียวคลุมหมด
]
for name, G, eg, er in tests:
    gb, rho = gamma_b(G), packing_number(G)
    print(f"{name}: γ_b={gb} (คาด {eg}), ρ={rho} (คาด {er})  {'✅' if gb==eg and rho==er else '❌'}")

P_4: γ_b=2 (คาด 2), ρ=2 (คาด 2)  ✅
C_6: γ_b=2 (คาด 2), ρ=2 (คาด 2)  ✅
K_4: γ_b=1 (คาด 1), ρ=1 (คาด 1)  ✅


In [3]:
import pandas as pd, math

def row(name, G):
    n    = G.number_of_nodes()
    degs = [d for _, d in G.degree()]
    delta = min(degs)
    gb, rho = gamma_b(G), packing_number(G)
    t = n // (delta + 1)
    r = {"graph": name, "n": n, "δ": delta, "ρ": rho, "γ_b": gb,
         "2ρ": 2*rho, "γ_b≤2ρ": "✅" if gb <= 2*rho else "❌",
         "2⌊n/(δ+1)⌋": 2*t, "γ_b≤2⌊n/(δ+1)⌋": "✅" if gb <= 2*t else "❌"}
    if len(set(degs)) == 1:                    # กราฟ r-regular → ทดสอบ Conjecture 3.8
        conj = math.ceil(n / (delta + 1))
        r["⌈n/(r+1)⌉"] = conj
        r["conj✅"] = "✅" if gb <= conj else "❌"
    return r

graphs  = [(f"P_{n}", nx.path_graph(n)) for n in range(4, 8)]
graphs += [(f"C_{n}", nx.cycle_graph(n)) for n in range(4, 9)]
graphs += [(f"S_{n}", nx.star_graph(n-1)) for n in range(5, 8)]
graphs += [("K_5", nx.complete_graph(5)),
           ("grid2x3", nx.grid_2d_graph(2, 3)),
           ("grid3x3", nx.grid_2d_graph(3, 3))]
graphs += [(f"Cin(1,2)_{n}", nx.circulant_graph(n, [1,2])) for n in range(6, 10)]
graphs += [(f"GP({n},1)", nx.circular_ladder_graph(n)) for n in range(4, 6)]

df = pd.DataFrame([row(name, G) for name, G in graphs])
print(df.to_string(index=False))

print("\n--- cross-check ค่าเป๊ะกับทฤษฎีบทของ paper ---")
for _, rw in df.iterrows():
    name = rw["graph"]
    if name.startswith("Cin"):
        n = int(name.split("_")[-1])
        print(f"{name}: γ_b={rw['γ_b']}  Thm3.5 ⌈n/5⌉={math.ceil(n/5)}  {'✅' if rw['γ_b']==math.ceil(n/5) else '❌'}")
    elif name.startswith("GP"):
        n = int(name[3])
        print(f"{name}: γ_b={rw['γ_b']}  Thm3.6 ⌈n/2⌉={math.ceil(n/2)}  {'✅' if rw['γ_b']==math.ceil(n/2) else '❌'}")

     graph  n  δ  ρ  γ_b  2ρ γ_b≤2ρ  2⌊n/(δ+1)⌋ γ_b≤2⌊n/(δ+1)⌋  ⌈n/(r+1)⌉ conj✅
       P_4  4  1  2    2   4      ✅           4              ✅        NaN   NaN
       P_5  5  1  2    2   4      ✅           4              ✅        NaN   NaN
       P_6  6  1  2    2   4      ✅           6              ✅        NaN   NaN
       P_7  7  1  3    3   6      ✅           6              ✅        NaN   NaN
       C_4  4  2  1    2   2      ✅           2              ✅        2.0     ✅
       C_5  5  2  1    2   2      ✅           2              ✅        2.0     ✅
       C_6  6  2  2    2   4      ✅           4              ✅        2.0     ✅
       C_7  7  2  2    3   4      ✅           4              ✅        3.0     ✅
       C_8  8  2  2    3   4      ✅           4              ✅        3.0     ✅
       S_5  5  1  1    1   2      ✅           4              ✅        NaN   NaN
       S_6  6  1  1    1   2      ✅           6              ✅        NaN   NaN
       S_7  7  1  1    1   2      ✅     